# TechOps Intelligence Platform
## Notebook 03 — PDF Processing Pipeline

**Author:** Sidhu  
**Phase:** 2 — Document Processing  
**Goal:** Extract, chunk, embed and store PDF content
          into ChromaDB knowledge_base collection

### PDFs Processed
- site_reliability_engineering.pdf (Google SRE Book)
- building_secure_and_reliable_systems.pdf (Google SRE Vol 2)
- sre_workbook.pdf (Google SRE Workbook)
- aws_well_architected.pdf (AWS Architecture Framework)
- aws_genai_lens.pdf (AWS GenAI Best Practices)

### Pipeline Steps
1. Load PDF with PyMuPDF
2. Detect if page is text-based or scanned
3. Extract text (direct or TrOCR fallback)
4. Clean and chunk by section
5. Embed with all-mpnet-base-v2
6. Store in ChromaDB with metadata
7. Test retrieval quality

In [1]:
# SETUP + IMPORTS
# ─────────────────────────────────────────
import os
import re
import json
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm

# PDF processing
import fitz  # PyMuPDF

# OCR for scanned pages
from PIL import Image
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel
)
import torch

# Embeddings
from sentence_transformers import SentenceTransformer

# Vector DB
import chromadb
# PROJECT PATHS
# ─────────────────────────────────────────
PROJECT_ROOT = Path("C:/Users/sudha/techops-intelligence")
os.chdir(PROJECT_ROOT)

RAW_PDFS    = PROJECT_ROOT / "data/raw/pdfs"
PROCESSED   = PROJECT_ROOT / "data/processed"
EMBEDDINGS  = PROJECT_ROOT / "data/embeddings"

## 1. Load Models
PyMuPDF for text extraction
TrOCR for scanned page fallback
all-mpnet-base-v2 for embeddings
ChromaDB for storage

In [2]:
# ─────────────────────────────────────────
# LOAD EMBEDDING MODEL
# Reuse same model as Notebook 02
# ─────────────────────────────────────────
print("Loading embedding model...")
embedding_model = SentenceTransformer(
    'sentence-transformers/all-mpnet-base-v2',
    device='cpu'
)
print("Embedding model loaded (768-dim)")

# ─────────────────────────────────────────
# LOAD TROCR FOR SCANNED PAGES
# Only loads when scanned page detected
# Lazy loading saves memory
# ─────────────────────────────────────────
trocr_processor = None
trocr_model     = None

def get_trocr():
    """Lazy load TrOCR — only when needed"""
    global trocr_processor, trocr_model
    if trocr_processor is None:
        print("Loading TrOCR for scanned page...")
        trocr_processor = TrOCRProcessor.from_pretrained(
            "microsoft/trocr-base-printed"
        )
        trocr_model = VisionEncoderDecoderModel.from_pretrained(
            "microsoft/trocr-base-printed"
        )
        trocr_model.eval()
        print("TrOCR loaded")
    return trocr_processor, trocr_model


# ─────────────────────────────────────────
# CONNECT TO CHROMADB
# Add to existing persistent store
# ─────────────────────────────────────────
chroma_path = str(EMBEDDINGS / "chroma_db")
client      = chromadb.PersistentClient(path=chroma_path)

# Create knowledge_base collection for PDFs
knowledge_base = client.get_or_create_collection(
    name     = "knowledge_base",
    metadata = {"hnsw:space": "cosine"}
)

print(f"\nChromaDB connected")
print(f"  knowledge_base collection: {knowledge_base.count()} docs")
print(f"  Existing collections: {[c.name for c in client.list_collections()]}")

Loading embedding model...
Embedding model loaded (768-dim)

ChromaDB connected
  knowledge_base collection: 11298 docs
  Existing collections: ['logs', 'playbooks', 'visuals', 'incidents', 'postmortems', 'knowledge_base']


## 2. PDF Processing Functions
Three-stage pipeline per page:
1. Try direct text extraction (fast)
2. If text < threshold → scanned page → TrOCR
3. Clean extracted text

In [3]:
# ─────────────────────────────────────────
# PDF PROCESSING FUNCTIONS
# ─────────────────────────────────────────

def is_scanned_page(page, min_chars: int = 50) -> bool:
    """
    Detect if a PDF page is scanned or text-based.
    Text-based pages have extractable text > min_chars.
    Scanned pages have little or no extractable text.
    """
    text = page.get_text().strip()
    return len(text) < min_chars


def extract_text_direct(page) -> str:
    """
    Extract text directly from text-based PDF page.
    Preserves structure better than raw get_text().
    """
    # Extract with layout preservation
    text = page.get_text("text")
    # Remove excessive whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()


def extract_text_ocr(page) -> str:
    """
    Extract text from scanned PDF page using TrOCR.
    Converts page to image first then runs OCR.
    """
    processor, model = get_trocr()

    # Render page as image at 200 DPI
    pix = page.get_pixmap(dpi=200)
    img = Image.frombytes(
        "RGB",
        [pix.width, pix.height],
        pix.samples
    )

    # TrOCR inference
    pixel_values = processor(
        img, return_tensors="pt"
    ).pixel_values

    with torch.no_grad():
        generated_ids = model.generate(pixel_values)

    text = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return text.strip()


def extract_pdf_text(pdf_path: str) -> list:
    """
    Extract text from all pages of a PDF.
    Returns list of page dicts with text + metadata.
    """
    doc    = fitz.open(pdf_path)
    pages  = []
    stats  = {"direct": 0, "ocr": 0, "empty": 0}

    for page_num in range(len(doc)):
        page = doc[page_num]

        if is_scanned_page(page):
            # Scanned page — use TrOCR
            try:
                text   = extract_text_ocr(page)
                method = "ocr"
                stats["ocr"] += 1
            except Exception as e:
                text   = ""
                method = "failed"
                stats["empty"] += 1
        else:
            # Text-based page — direct extraction
            text   = extract_text_direct(page)
            method = "direct"
            stats["direct"] += 1

        if text.strip():
            pages.append({
                "page_num"   : page_num + 1,
                "text"       : text,
                "method"     : method,
                "char_count" : len(text),
                "pdf_path"   : str(pdf_path)
            })

    doc.close()
    return pages, stats


def clean_pdf_text(text: str) -> str:
    """
    Clean extracted PDF text.
    Removes headers, footers, page numbers.
    Preserves technical content.
    """
    # Remove page numbers (standalone numbers)
    text = re.sub(r'^\d+$', '', text, flags=re.MULTILINE)

    # Remove common PDF artifacts
    text = re.sub(r'\x0c', '\n', text)  # form feed
    text = re.sub(r'\ufeff', '', text)  # BOM

    # Normalise whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)

    # Remove very short lines (likely headers/footers)
    lines       = text.split('\n')
    clean_lines = [
        l for l in lines
        if len(l.strip()) > 3 or l.strip() == ''
    ]
    text = '\n'.join(clean_lines)

    return text.strip()


print("PDF processing functions defined")

# Quick test on first PDF
test_pdf = list(RAW_PDFS.glob("*.pdf"))[0]
print(f"\nTesting on: {test_pdf.name}")
doc = fitz.open(str(test_pdf))
page = doc[5]  # page 6 (skip cover)
is_scanned = is_scanned_page(page)
text_preview = extract_text_direct(page)[:200]
doc.close()

print(f"Is scanned     : {is_scanned}")
print(f"Text preview   :\n{text_preview}")

PDF processing functions defined

Testing on: aws_genai_lens.pdf
Is scanned     : False
Text preview   :
Generative AI Lens
AWS Well-Architected Framework
GENOPS05-BP01 Learn when to customize models ................................................................... 118
Security ........................


## 3. Chunking Strategy
Section-aware chunking for SRE content:
→ Split on chapter/section headings
→ Fallback to RecursiveCharacterTextSplitter
→ Chunk size: 500 chars, overlap: 50
→ Preserve heading context in each chunk

In [4]:
# ─────────────────────────────────────────
# CHUNKING STRATEGY
# Section-aware for SRE/AWS documents
# ─────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Section heading patterns for SRE content
SECTION_PATTERNS = [
    r'^Chapter \d+',
    r'^\d+\.\d+\s+[A-Z]',    # "1.2 Section Title"
    r'^[A-Z][A-Z\s]{10,}$',  # "ALL CAPS HEADING"
]

def split_into_chunks(
    text     : str,
    chunk_size: int = 700,
    overlap   : int = 100
) -> list:
    """
    Split text into chunks for embedding.
    Uses RecursiveCharacterTextSplitter with
    paragraph-aware splitting.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size         = chunk_size,
        chunk_overlap      = overlap,
        separators         = [
            "\n\n",   # paragraph break first
            "\n",     # line break
            ". ",     # sentence break
            " ",      # word break
            ""        # character (last resort)
        ],
        length_function    = len,
        is_separator_regex = False
    )
    chunks = splitter.split_text(text)
    # Filter very short chunks
    chunks = [c for c in chunks if len(c.strip()) > 50]
    return chunks


def process_pdf_into_chunks(
    pdf_path    : Path,
    chunk_size  : int = 700,
    overlap     : int = 100
) -> list:
    """
    Full pipeline: PDF → pages → clean → chunks
    Returns list of chunk dicts ready for embedding
    """
    print(f"\nProcessing: {pdf_path.name}")
    pages, stats = extract_pdf_text(str(pdf_path))
    print(f"  Pages extracted : {len(pages)}")
    print(f"  Method breakdown: {stats}")

    all_chunks = []
    for page in pages:
        clean_text = clean_pdf_text(page['text'])
        if len(clean_text) < 50:
            continue

        chunks = split_into_chunks(
            clean_text,
            chunk_size,
            overlap
        )

        for j, chunk in enumerate(chunks):
            all_chunks.append({
                "text"    : chunk,
                "page_num": page['page_num'],
                "method"  : page['method'],
                "source"  : pdf_path.name,
                "chunk_id": j
            })

    print(f"  Total chunks    : {len(all_chunks)}")
    total_chars = sum(len(c['text']) for c in all_chunks)
    print(f"  Total chars     : {total_chars:,}")
    print(f"  Avg chunk size  : {total_chars//max(len(all_chunks),1)} chars")

    return all_chunks


# Test chunking on one PDF
test_chunks = process_pdf_into_chunks(
    RAW_PDFS / "aws_genai_lens.pdf"
)
print(f"\nSample chunk:")
print(f"  {test_chunks[10]['text'][:300]}")


Processing: aws_genai_lens.pdf
  Pages extracted : 261
  Method breakdown: {'direct': 261, 'ocr': 0, 'empty': 0}
  Total chunks    : 974
  Total chars     : 568,387
  Avg chunk size  : 583 chars

Sample chunk:
  Hybrids .......................................................................................................................................................... 23
Conclusion ...........................................................................................................................


## 4. Process All PDFs
Extract, clean, chunk all 5 PDFs
Store in ChromaDB knowledge_base collection

In [5]:
# ─────────────────────────────────────────
# PROCESS ALL PDFs — SKIP PDFs ALREADY IN CHROMADB
# ─────────────────────────────────────────
CHUNK_SIZE = 700
OVERLAP    = 100
BATCH_SIZE = 32


def existing_embedding_count(collection, pdf_name: str) -> int:
    """Return the number of embeddings already stored for one PDF."""
    existing = collection.get(
        where={"pdf_name": pdf_name},
        include=["metadatas"]
    )
    return len(existing["ids"])


def embed_and_store_chunks(
    chunks: list,
    collection,
    model,
    pdf_name: str,
    batch_size: int = 32
):
    """Embed PDF chunks and store them in ChromaDB."""
    texts = [chunk["text"] for chunk in chunks]

    ids = [
        f"pdf_{pdf_name}_{chunk['page_num']}_{chunk['chunk_id']}"
        for chunk in chunks
    ]

    metadatas = [
        {
            "source": chunk["source"],
            "page_num": str(chunk["page_num"]),
            "method": chunk["method"],
            "doc_type": "pdf_knowledge",
            "pdf_name": pdf_name
        }
        for chunk in chunks
    ]

    stored = 0
    errors = 0

    for i in tqdm(
        range(0, len(texts), batch_size),
        desc=f"Embedding {pdf_name}"
    ):
        batch_texts = texts[i:i + batch_size]
        batch_ids = ids[i:i + batch_size]
        batch_metadata = metadatas[i:i + batch_size]

        valid = [
            (text, chunk_id, metadata)
            for text, chunk_id, metadata in zip(
                batch_texts,
                batch_ids,
                batch_metadata
            )
            if text and len(text.strip()) > 50
        ]

        if not valid:
            continue

        valid_texts, valid_ids, valid_metadata = zip(*valid)

        try:
            embeddings = model.encode(
                list(valid_texts),
                show_progress_bar=False
            )

            collection.upsert(
                documents=list(valid_texts),
                embeddings=embeddings.tolist(),
                metadatas=list(valid_metadata),
                ids=list(valid_ids)
            )

            stored += len(valid_texts)

        except Exception as error:
            errors += 1
            print(f"  Batch error: {error}")

    return stored, errors


# ── Process all PDFs ─────────────────────
pdf_files = sorted(RAW_PDFS.glob("*.pdf"))
all_results = {}

for pdf_path in pdf_files:
    pdf_key = pdf_path.stem  # Same value saved as metadata: pdf_name

    # Check ChromaDB before extracting or embedding
    existing_count = existing_embedding_count(
        knowledge_base,
        pdf_key
    )

    if existing_count > 0:
        all_results[pdf_path.name] = {
            "chunks": existing_count,
            "stored": existing_count,
            "errors": 0,
            "status": "skipped"
        }

        print(
            f"Skipping {pdf_path.name} "
            f"({existing_count} embeddings already exist)"
        )
        continue

    # Extract and chunk only new PDFs
    chunks = process_pdf_into_chunks(
        pdf_path,
        CHUNK_SIZE,
        OVERLAP
    )

    if not chunks:
        print(f"No chunks extracted from {pdf_path.name}")
        continue

    # Embed and save new PDF chunks
    stored, errors = embed_and_store_chunks(
        chunks=chunks,
        collection=knowledge_base,
        model=embedding_model,
        pdf_name=pdf_key,
        batch_size=BATCH_SIZE
    )

    all_results[pdf_path.name] = {
        "chunks": len(chunks),
        "stored": stored,
        "errors": errors,
        "status": "processed"
    }

    print(f"{pdf_path.name}: {stored} chunks stored")


# ── Summary ──────────────────────────────
print("\n" + "=" * 65)
print("PDF PROCESSING SUMMARY")
print("=" * 65)

for pdf_name, result in all_results.items():
    print(
        f"  {pdf_name:50} → "
        f"{result['stored']:4} chunks "
        f"({result['status']})"
    )

print(f"\nTotal in knowledge_base: {knowledge_base.count():,}")

Skipping aws_genai_lens.pdf (974 embeddings already exist)
Skipping aws_well_architected.pdf (3600 embeddings already exist)
Skipping building_secure_and_reliable_systems.pdf (2289 embeddings already exist)

Processing: high_performance_sre.pdf
Loading TrOCR for scanned page...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TrOCR loaded
  Pages extracted : 275
  Method breakdown: {'direct': 269, 'ocr': 6, 'empty': 0}
  Total chunks    : 853
  Total chars     : 488,202
  Avg chunk size  : 572 chars


Embedding high_performance_sre: 100%|██████████| 27/27 [02:13<00:00,  4.93s/it]

high_performance_sre.pdf: 853 chunks stored
Skipping site_reliability_engineering.pdf (2050 embeddings already exist)
Skipping sre_workbook.pdf (1915 embeddings already exist)

PDF PROCESSING SUMMARY
  aws_genai_lens.pdf                                 →  974 chunks (skipped)
  aws_well_architected.pdf                           → 3600 chunks (skipped)
  building_secure_and_reliable_systems.pdf           → 2289 chunks (skipped)
  high_performance_sre.pdf                           →  853 chunks (processed)
  site_reliability_engineering.pdf                   → 2050 chunks (skipped)
  sre_workbook.pdf                                   → 1915 chunks (skipped)

Total in knowledge_base: 12,151


## 5. Retrieval Quality Test
Test that PDF knowledge is retrievable
for diagnosis and resolution agents

In [6]:
# ─────────────────────────────────────────
# PDF KNOWLEDGE RETRIEVAL TEST
# Queries that resolution agent will make
# ─────────────────────────────────────────

def pdf_search(query, collection, model, n=3):
    query_embedding = model.encode([query]).tolist()
    results = collection.query(
        query_embeddings = query_embedding,
        n_results        = n,
        include          = ['documents', 'metadatas', 'distances']
    )
    return results


pdf_queries = [
    "how to handle on-call incidents and escalation",
    "postmortem blameless culture lessons learned",
    "error budget SLO SLA reliability",
    "load balancing traffic management reliability",
    "generative AI security risks guardrails",
    "AWS well architected reliability pillar",
    "incident response runbook troubleshooting steps",
    "cascading failures circuit breaker pattern"
]

print("=== PDF Knowledge Retrieval Test ===\n")
scores = []

for query in pdf_queries:
    results = pdf_search(
        query,
        knowledge_base,
        embedding_model,
        n=2
    )

    docs      = results['documents'][0]
    distances = results['distances'][0]
    metas     = results['metadatas'][0]

    top_score = 1 - distances[0]
    scores.append(top_score)

    print(f"Query : {query}")
    print(f"Score : {top_score:.3f} | PDF: {metas[0].get('source','')}")
    print(f"Text  : {docs[0][:120]}...")
    print()

print(f"=== Score Summary ===")
print(f"Min score : {min(scores):.3f}")
print(f"Max score : {max(scores):.3f}")
print(f"Avg score : {np.mean(scores):.3f}")

=== PDF Knowledge Retrieval Test ===



Query : how to handle on-call incidents and escalation
Score : 0.762 | PDF: aws_well_architected.pdf
Text  : 2. Set up on-call schedules: Create on-call schedules in Incident Manager that align with your 
escalation paths. Equip ...

Query : postmortem blameless culture lessons learned
Score : 0.718 | PDF: sre_workbook.pdf
Text  : Model and Enforce Blameless Behavior
To properly support postmortem culture, engineering leaders should consistently
exe...

Query : error budget SLO SLA reliability
Score : 0.726 | PDF: high_performance_sre.pdf
Text  : define the error budget. If your SLA guarantees an availability of 99.99%, your
error budget is 0.01%. It indicates the ...

Query : load balancing traffic management reliability
Score : 0.641 | PDF: sre_workbook.pdf
Text  : availability and reliability over traditional load balancing systems (which typi‐
cally rely on active/passive pairs to ...

Query : generative AI security risks guardrails
Score : 0.709 | PDF: aws_genai_lens.pdf
Text  : G

In [7]:
# ─────────────────────────────────────────
# NOTEBOOK 03 — FINAL SUMMARY
# ─────────────────────────────────────────
print("   NOTEBOOK 03 — PDF PIPELINE COMPLETE")
print("=" * 55)

print(f"\n  PDFs processed : {len(all_results)}")
for pdf_name, result in all_results.items():
    print(f"  {pdf_name:50} → {result['stored']:4} chunks")

print(f"\n  knowledge_base total : {knowledge_base.count():,} chunks")
print(f"  Chunk size           : {CHUNK_SIZE} chars")
print(f"  Chunk overlap        : {OVERLAP} chars")
print(f"  Embedding model      : all-mpnet-base-v2 (768-dim)")
print(f"  Avg retrieval score  : {np.mean(scores):.3f}")
print(f"  Min retrieval score  : {min(scores):.3f}")
print(f"  Max retrieval score  : {max(scores):.3f}")


   NOTEBOOK 03 — PDF PIPELINE COMPLETE

  PDFs processed : 6
  aws_genai_lens.pdf                                 →  974 chunks
  aws_well_architected.pdf                           → 3600 chunks
  building_secure_and_reliable_systems.pdf           → 2289 chunks
  high_performance_sre.pdf                           →  853 chunks
  site_reliability_engineering.pdf                   → 2050 chunks
  sre_workbook.pdf                                   → 1915 chunks

  knowledge_base total : 12,151 chunks
  Chunk size           : 700 chars
  Chunk overlap        : 100 chars
  Embedding model      : all-mpnet-base-v2 (768-dim)
  Avg retrieval score  : 0.700
  Min retrieval score  : 0.506
  Max retrieval score  : 0.804


In [8]:
# ─────────────────────────────────────────
# ADD HIGH PERFORMANCE SRE PDF
# Anchal Arora Mishra — Walmart SRE
# Covers: SLOs, error budgets, incident
# management, automation, on-call ops
# ─────────────────────────────────────────
new_pdf = RAW_PDFS / "high_performance_sre.pdf"

if not new_pdf.exists():
    print("high_performance_sre.pdf not found in data/raw/pdfs/")
    print("Place the PDF there and re-run this cell")
else:
    # Check if already embedded
    existing      = knowledge_base.get(include=[])
    existing_ids  = set(existing['ids'])
    already_done  = any(
        "high_performance_sre" in id_
        for id_ in existing_ids
    )

    if already_done:
        print("high_performance_sre.pdf already embedded - skipping")
    else:
        print(f"Processing: {new_pdf.name}")

        chunks = process_pdf_into_chunks(
            new_pdf,
            chunk_size = 700,
            overlap    = 100
        )

        if chunks:
            stored, errors = embed_and_store_chunks(
                chunks,
                knowledge_base,
                embedding_model,
                new_pdf.stem,
                BATCH_SIZE
            )
            print(f"Stored: {stored} chunks")
            print(f"Errors: {errors}")
        else:
            print("No chunks extracted - check if PDF is readable")

    print(f"\nknowledge_base total: {knowledge_base.count():,} chunks")

high_performance_sre.pdf already embedded - skipping

knowledge_base total: 12,151 chunks
